In [ ]:
# wandb
%pip install wandb
import wandb
wandb.login()

In [ ]:
# cv2.dnn installsion issue
#%pip install opencv-python==4.8.0.74
# import libraries
import cv2
import torch
import pytorch_lightning as pl
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, datasets
from torchmetrics import Accuracy


import torchmetrics
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np

import os
from pytorch_lightning.loggers import WandbLogger

In [ ]:
# 설정
# img size and path
IMG_SIZE = 300
PATH = "../../data/dog_eye_for_train/imgs/"
TEST_PATH = "../../data/dog_eye_for_test/imgs/"
CLASSES = 11 # ten symtoms + normal
BATCH_SIZE = 512

In [ ]:
# CNN model class
class PrettyEyes(pl.LightningModule):
    def __init__(self):
        super().__init__()
        
        ### layers
        self.conv1 = nn.Conv2d(3, 16, 3, 1) # in_channel, out_channel, kernel, stride # (1, 16, 298, 298)
        self.pool1 = nn.MaxPool2d(2) # (1, 16, 149, 149)
        self.conv2 = nn.Conv2d(16, 32, 3, 1) # (1, 32, 147, 147)
        self.pool2 = nn.MaxPool2d(2) # (1, 32, 74, 74)
        self.conv3 = nn.Conv2d(32, 64, 3, 1) # (1, 64, 72, 72)
        self.pool3 = nn.MaxPool2d(2) # (1, 64, 36, 36)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(78400, 512)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(512, CLASSES) 
        
        ### metric
        self.accuracy = Accuracy(num_classes=CLASSES,task='multiclass')
        
        ### for epoch-wise logging
        self.train_accuracy = self.accuracy
        self.val_accuracy = self.accuracy

    def forward(self, x):
        x = self.pool1(self.conv1(x))
        x = self.pool2(self.conv2(x))
        x = self.pool3(self.conv3(x))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

    def loss_function(self, out, target):
        loss = nn.CrossEntropyLoss()(out, target)
        return loss
    
    def configure_optimizers(self):
        lr = 0.001
        optimizer = torch.optim.Adam(self.parameters(), lr=lr)
        return optimizer
    
    def training_step(self, batch, batch_idx):
        x, y = batch['x'], batch['y']
        img = x.view(-1, 3, IMG_SIZE, IMG_SIZE)
        label = y.view(-1)
        out = self(img)
        loss = self.loss_function(out, label)
        logits = torch.argmax(out, dim=1)
        acc = self.accuracy(logits, label)
        
        self.log('train_loss', loss, on_step=True, on_epoch=False)
        self.log('train_acc', acc, on_step=True, on_epoch=False)
        
        self.train_accuracy(logits, label)
        
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch['x'], batch['y']
        img = x.view(-1, 3, IMG_SIZE, IMG_SIZE)
        label = y.view(-1)
        out = self(img)
        loss = self.loss_function(out, label)
        logits = torch.argmax(out, dim=1)
        acc = self.accuracy(logits, label)
        
        self.log('val_loss', loss, on_step=True, on_epoch=True)
        self.log('val_acc', acc, on_step=True, on_epoch=False)
        
        self.val_accuracy(logits, label)
        
        return loss

    def on_train_epoch_end(self):
        train_acc_epoch = self.train_accuracy.compute()
        self.log('train_acc_epoch', train_acc_epoch, on_step=False, on_epoch=True)
        self.train_accuracy.reset()
    
    def on_validation_epoch_end(self):
        val_acc_epoch = self.val_accuracy.compute()
        self.log('val_acc_epoch', val_acc_epoch, on_step=False, on_epoch=True)
        self.val_accuracy.reset()

In [ ]:
# Dataset class
class DogEyes(Dataset):
    def __init__(self, path, img_ids, labels, img_size):
        self.img_ids = img_ids
        self.labels = labels
        self.path = path
        self.img_size = img_size

    def __len__(self):
        return len(self.img_ids)
    
    def __getitem__(self, item):
        img_ids = str(self.img_ids[item])
        labels = self.labels[item]
        img_file = cv2.imread(self.path+img_ids)
        img = cv2.resize(img_file, (self.img_size, self.img_size))
        img = img.astype(np.float64)
        item_dic = {
            'x': torch.tensor(img, dtype=torch.float),
            'y': torch.tensor(labels, dtype=torch.long)
            }
        return item_dic

In [ ]:
# DataLoader
class DogEyesLoader(pl.LightningDataModule):
    def __init__(self, batch_size=512):
        super().__init__()
        self.batch_size = batch_size

    def setup(self, stage=None):
        df = pd.read_csv('../../data/dog_eye_for_train/csv/dog_eyes.csv')
        x_train, x_val, y_train, y_val = train_test_split(df['img_id'].values, df.label.values, test_size=0.1)
        self.train_dataset = DogEyes(PATH, x_train, y_train, IMG_SIZE)
        self.val_dataset = DogEyes(PATH, x_val, y_val, IMG_SIZE)

    def train_dataloader(self):
        train_loader = DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)
        return train_loader
    
    def val_dataloader(self):
        val_loader = DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)
        return val_loader

In [ ]:
# Callback
callback = pl.callbacks.ModelCheckpoint(
    monitor='val_loss',
    dirpath='../../data/dog_eye_for_train/models/',
    filename='models-{epoch:02d}-{val_loss:.2f}',
    save_top_k=3,
    mode='min'
)

In [ ]:
# Train
model = PrettyEyes()
datamodule = DogEyesLoader()
wandb_logger = WandbLogger(project='forest-cv')

trainer = pl.Trainer(accelerator='auto', max_epochs=10, log_every_n_steps=13, callbacks=[callback],  logger=wandb_logger)
trainer.fit(model=model, datamodule=datamodule)

In [ ]:
# Predict Dataset class
class DogEyesPred(Dataset):
    def __init__(self, path, img_ids, img_size):
        self.img_ids = img_ids
        self.path = path
        self.img_size = img_size

    def __len__(self):
        return len(self.img_ids)
    
    def __getitem__(self, item):
        img_ids = str(self.img_ids[item])
        img_file = cv2.imread(self.path+img_ids)
        img = cv2.resize(img_file, (self.img_size, self.img_size))
        img = img.astype(np.float64)
        img = torch.tensor(img, dtype=torch.float).view(3, IMG_SIZE, IMG_SIZE)

        return img 

In [ ]:
# Predict
best_model_path = '../../data/dog_eye_for_train/models/models_0521/models-epoch=15-val_loss=0.79.ckpt'
model = PrettyEyes.load_from_checkpoint(best_model_path)

trainer = pl.Trainer()
pred_data = pd.read_csv('../../data/dog_eye_for_test/csv/dog_eyes.csv')
pred_dataset = DogEyesPred(TEST_PATH, pred_data['img_id'].values, IMG_SIZE)
predict_dataloader = DataLoader(pred_dataset, shuffle=False)

pred = trainer.predict(model=model, dataloaders=predict_dataloader)
pred = [torch.argmax(p, dim=1)[0] for p in pred]

pred_df = pd.DataFrame(pred, columns=['prediction'])
pred_df.to_csv('../../data/dog_eye_for_test/csv/pred_result.csv', index=False)